# Character Creation Workflow

This notebook demonstrates the complete character creation process for D&D 5e characters, from basic info to Neo4j persistence.


In [ ]:
import sys
from pathlib import Path
from uuid import uuid4

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from rpg_graph_vtt.graph.connection import get_connection
from rpg_graph_vtt.graph.queries import CharacterQueries, GameSystemQueries
from rpg_graph_vtt.models.character import Character, CharacterCreate, AbilityScores

# Connect to Neo4j
conn = get_connection()
print("Connected to Neo4j")


## Step 1: View Available Options

First, let's see what classes, races, and backgrounds are available.


In [ ]:
# Get available classes
classes = GameSystemQueries.get_all_classes(conn)
print("Available Classes:")
for cls in classes[:5]:  # Show first 5
    print(f"  - {cls['name']} (Hit Die: d{cls['hit_die']}, Primary: {cls['primary_ability']})")

# Get available races
races = GameSystemQueries.get_all_races(conn)
print("\nAvailable Races:")
for race in races[:5]:  # Show first 5
    print(f"  - {race['name']} (Size: {race['size']}, Speed: {race['speed']}ft)")


## Step 2: Create a Character

Let's create a sample character: a 1st level Human Fighter.


In [ ]:
# Character creation data
character_name = "Aragorn"
character_level = 1
class_name = "Fighter"
race_name = "Human"
background_name = "Soldier"

# Standard array ability scores (15, 14, 13, 12, 10, 8)
ability_scores = AbilityScores(
    strength=15,
    dexterity=13,
    constitution=14,
    intelligence=12,
    wisdom=10,
    charisma=8
)

# Calculate modifiers
modifiers = ability_scores.get_modifiers()

# Calculate proficiency bonus (based on level)
proficiency_bonus = (character_level - 1) // 4 + 2

# Calculate hit points (assuming Fighter with CON modifier)
con_mod = modifiers["con"]
hit_die = 10  # Fighter hit die
hit_points_max = hit_die + con_mod
hit_points = hit_points_max

# Calculate armor class (assuming chain mail: AC 16)
armor_class = 16

# Create character object
character = Character(
    uuid=uuid4(),
    name=character_name,
    level=character_level,
    hit_points=hit_points,
    hit_points_max=hit_points_max,
    armor_class=armor_class,
    proficiency_bonus=proficiency_bonus,
    ability_scores=ability_scores,
    ability_modifiers=modifiers,
    saving_throws={"str": True, "con": True},  # Fighter saving throw proficiencies
    skills={"athletics": True, "intimidation": True}  # Example skill proficiencies
)

print(f"Created character: {character.name}")
print(f"  Level: {character.level}")
print(f"  Class: {class_name}")
print(f"  Race: {race_name}")
print(f"  HP: {character.hit_points}/{character.hit_points_max}")
print(f"  AC: {character.armor_class}")
print(f"  Proficiency Bonus: +{character.proficiency_bonus}")


## Step 3: Persist to Neo4j

Now we'll save the character to Neo4j and create all the relationships.


In [ ]:
# Create character node
character_data = character.to_neo4j_dict()
character_uuid = CharacterQueries.create_character(character_data, conn)
print(f"✓ Character created with UUID: {character_uuid}")

# Link to class
CharacterQueries.link_character_to_class(character_uuid, class_name, character_level, conn)
print(f"✓ Linked to class: {class_name}")

# Link to race
CharacterQueries.link_character_to_race(character_uuid, race_name, conn)
print(f"✓ Linked to race: {race_name}")

# Link to background
CharacterQueries.link_character_to_background(character_uuid, background_name, conn)
print(f"✓ Linked to background: {background_name}")

print("\n✓ Character fully created and linked in Neo4j!")


## Step 4: Retrieve and Verify

Let's retrieve the character from Neo4j to verify everything was saved correctly.


In [ ]:
# Get character with all relationships
character_data = CharacterQueries.get_character_with_relationships(character_uuid, conn)

if character_data:
    print(f"Character: {character_data['name']}")
    print(f"  Level: {character_data['level']}")
    print(f"  HP: {character_data['hit_points']}/{character_data['hit_points_max']}")
    print(f"  AC: {character_data['armor_class']}")
    
    if character_data.get('classes'):
        print(f"  Classes: {', '.join([c['name'] for c in character_data['classes']])}")
    if character_data.get('races'):
        print(f"  Races: {', '.join([r['name'] for r in character_data['races']])}")
    if character_data.get('backgrounds'):
        print(f"  Backgrounds: {', '.join([b['name'] for b in character_data['backgrounds']])}")
else:
    print("Character not found")
